In [1]:
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
import time

print("PyTorch版本:", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"当前设备: {device}")

PyTorch版本: 2.11.0+cu130
当前设备: cuda


In [2]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Cell 3: 加载数据集
data_dir = './data/PlantVillage'
full_dataset = datasets.ImageFolder(root=data_dir, transform=train_transform)
class_names = full_dataset.classes
num_classes = len(class_names)
print(f"类别: {class_names}")
print(f"总图片数: {len(full_dataset)}")

类别: ['Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___healthy']
总图片数: 6625


In [3]:
# 划分数据集
torch.manual_seed(42)  # 固定随机种子，确保结果可重复
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])
val_dataset.dataset.transform = test_transform

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"训练集: {train_size} 张")
print(f"验证集: {val_size} 张")

训练集: 5300 张
验证集: 1325 张


In [4]:
# Cell 4: 通用训练函数
def train_model(model, criterion, optimizer, epochs=5, name=""):
    """通用训练函数"""
    train_losses = []
    val_accs = []
    start_time = time.time()
    
    for epoch in range(epochs):
        # 训练阶段
        model.train()
        running_loss = 0.0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
        
        # 验证阶段
        model.eval()
        correct = 0
        total = 0
        
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = outputs.max(1)
                correct += predicted.eq(labels).sum().item()
                total += labels.size(0)
        
        avg_loss = running_loss / len(train_loader)
        val_acc = 100.0 * correct / total
        train_losses.append(avg_loss)
        val_accs.append(val_acc)
        
        print(f"{name} Epoch [{epoch+1}/{epochs}] | Loss: {avg_loss:.4f} | Val Acc: {val_acc:.2f}%")
    
    elapsed = time.time() - start_time
    best_acc = max(val_accs)
    print(f"{name} 完成！用时: {elapsed:.1f}s, 最佳准确率: {best_acc:.2f}%")
    
    return train_losses, val_accs, elapsed, best_acc


In [5]:
# 方案1 - 从头训练（随机初始化）
print("\n" + "="*50)
print("方案1：从头训练（随机初始化）")
print("="*50)

model_scratch = models.resnet18(weights=None)  # 随机初始化
model_scratch.fc = nn.Linear(model_scratch.fc.in_features, num_classes)
model_scratch = model_scratch.to(device)

criterion = nn.CrossEntropyLoss()
optimizer_scratch = optim.Adam(model_scratch.parameters(), lr=0.001)

loss_scratch, acc_scratch, time_scratch, best_scratch = train_model(
    model_scratch, criterion, optimizer_scratch, epochs=5, name="方案1(从头训练)"
)

# 保存方案1的结果
results = {
    'scheme1': {'loss': loss_scratch, 'acc': acc_scratch, 'time': time_scratch, 'best': best_scratch}
}

print(f"\n方案1完成！最佳准确率: {best_scratch:.2f}%")


方案1：从头训练（随机初始化）
方案1(从头训练) Epoch [1/5] | Loss: 0.5550 | Val Acc: 74.57%
方案1(从头训练) Epoch [2/5] | Loss: 0.3394 | Val Acc: 71.32%
方案1(从头训练) Epoch [3/5] | Loss: 0.2686 | Val Acc: 89.89%
方案1(从头训练) Epoch [4/5] | Loss: 0.2427 | Val Acc: 82.19%
方案1(从头训练) Epoch [5/5] | Loss: 0.2158 | Val Acc: 90.34%
方案1(从头训练) 完成！用时: 318.8s, 最佳准确率: 90.34%

方案1完成！最佳准确率: 90.34%


In [6]:
# Cell 6: 方案2 - 冻结特征提取
print("\n" + "="*50)
print("方案2：冻结特征提取")
print("="*50)

model_frozen = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# 冻结所有参数
for param in model_frozen.parameters():
    param.requires_grad = False

# 替换分类头
model_frozen.fc = nn.Linear(model_frozen.fc.in_features, num_classes)
model_frozen = model_frozen.to(device)

# 只优化分类头
optimizer_frozen = optim.Adam(model_frozen.fc.parameters(), lr=0.001)

loss_frozen, acc_frozen, time_frozen, best_frozen = train_model(
    model_frozen, criterion, optimizer_frozen, epochs=5, name="方案2(冻结特征)"
)

# 保存方案2的结果
results['scheme2'] = {'loss': loss_frozen, 'acc': acc_frozen, 'time': time_frozen, 'best': best_frozen}

print(f"\n方案2完成！最佳准确率: {best_frozen:.2f}%")


方案2：冻结特征提取
方案2(冻结特征) Epoch [1/5] | Loss: 0.5418 | Val Acc: 93.74%
方案2(冻结特征) Epoch [2/5] | Loss: 0.2685 | Val Acc: 94.04%
方案2(冻结特征) Epoch [3/5] | Loss: 0.2120 | Val Acc: 94.79%
方案2(冻结特征) Epoch [4/5] | Loss: 0.1875 | Val Acc: 96.15%
方案2(冻结特征) Epoch [5/5] | Loss: 0.1653 | Val Acc: 94.87%
方案2(冻结特征) 完成！用时: 316.6s, 最佳准确率: 96.15%

方案2完成！最佳准确率: 96.15%
